In [ ]:
# Cài đặt Ollama, zstd và các thư viện Python cần thiết
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install -y zstd  # Lệnh bổ sung để sửa lỗi giải nén
!curl -fsSL https://ollama.com/install.sh | sh
!pip install fastapi nest-asyncio uvicorn omegaconf requests pydantic

import os
import subprocess
import threading
import time

# Khởi chạy Ollama ngầm ở cổng 11434
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0"
env["OLLAMA_ORIGINS"] = "*"

def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"], env=env)

thread = threading.Thread(target=run_ollama_serve)
thread.start()

# Đợi 5 giây để máy chủ khởi động hoàn toàn
time.sleep(5)
print("Ollama server đã khởi động thành công!")

In [ ]:
# Tải mô hình Qwen về local
!ollama pull qwen2.5:0.5b

In [ ]:
yaml_config = """
model_name: "qwen2.5:0.5b"
ollama_url: "http://localhost:11434/api/generate"
"""
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)
print("Đã tạo file config.yaml")

In [ ]:
import requests
from omegaconf import OmegaConf

class QwenGeneration:
    def __init__(self, config_path):
        # Đọc cấu hình từ file yaml
        self.config = OmegaConf.load(config_path)
        self.model_name = self.config.model_name
        self.url = self.config.ollama_url

    def __call__(self, prompt):
        # Đóng gói dữ liệu và gửi sang Ollama xử lý
        payload = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": False
        }
        try:
            response = requests.post(self.url, json=payload, timeout=120)
            response.raise_for_status()
            data = response.json()
            return data["response"]
        except Exception as e:
            return f"Lỗi khi xử lý mô hình: {str(e)}"

# Khởi tạo ngay một object từ Class để chuẩn bị cho FastAPI
qwen_model = QwenGeneration("config.yaml")

In [ ]:
import nest_asyncio
import uvicorn
import threading
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

# Khai báo cấu trúc dữ liệu đầu vào
class InferenceRequest(BaseModel):
    prompt: str

# Endpoint 1: Giới thiệu [cite: 38]
@app.get('/')
async def root():
    return {"message": "Hệ thống Web API sử dụng Qwen model thông qua Ollama & FastAPI"}

# Endpoint 2: Kiểm tra trạng thái [cite: 39]
@app.get('/health')
async def health_check():
    return {"status": "API is up and running"}

# Endpoint 3: Sinh văn bản [cite: 40]
@app.post('/generate')
async def generate_text(req: InferenceRequest):
    if not req.prompt:
        raise HTTPException(status_code=400, detail="Prompt không được để trống")

    # Kích hoạt phương thức __call__ của object chuẩn OOP
    result = qwen_model(req.prompt)

    return {"result": result}

# Chạy máy chủ FastAPI ngầm ở cổng 8000
nest_asyncio.apply()

def run_fastapi():
    uvicorn.run(app, host="0.0.0.0", port=8000)

fastapi_thread = threading.Thread(target=run_fastapi, daemon=True)
fastapi_thread.start()
print("FastAPI Server đã khởi chạy tại: http://0.0.0.0:8000")

In [ ]:
# Lệnh này sẽ sinh ra một link dạng https://xxxx.a.free.pinggy.link
# LƯU Ý: Không nhấn Ctrl+C, nếu muốn dừng thì dừng Cell thủ công
!ssh -T -o StrictHostKeyChecking=no -p 443 -R0:localhost:8000 a.pinggy.io